# B2 context extension (896 → 1024) + continued pretraining on Kaggle 2×T4 (resumable)

**Do not start full training until the data manifest, contamination checks and this configuration have been reviewed.**
Session 1 runs the **probe only** (`RUN_TRAINING = False`): speed, memory and the session count for your token budget.

**Kaggle settings:** Accelerator **GPU T4 ×2**; internet not needed.
**Add Input:**
1. the output of `capstone_b2_ctxdata` (CPU data audit): `ctxdata/new_*.bin`, `ctxdata/heldout_*.bin`, `manifest_ctxext.json`;
2. the output of your Phase-1 **data** notebook: the original `train.bin`, `val.bin` (SC validation);
3. your Phase-1 **B2** output: `run_B2/best.pt` (read-only input: the original checkpoint is never modified);
4. from session 2 on: the previous session's output of THIS notebook (`ctxext/last.pt`). A private notebook's output is not
   visible to another account: on the old account use *Output → New Dataset* (private) and share it with the new account.

**Sampler:** fixed non-overlapping 1024-token windows, each used at most once, in a seeded order saved in `last.pt`
(`nr_sampler.py`); a resumed session continues at the saved cursor.

**Session rules (enforced by the script):** the same number of GPUs (2) and the same CONFIG in every session. A resume that
would change world size, batch tokens, tokens/step, token target, length, RoPE method, LR or schedule stops with an error.

Outputs in `/kaggle/working/ctxext/`:
- `last.pt`: full resumable state;
- `best_newctx.pt`: lowest mean 1024-token loss;
- `best_oldctx.pt`: lowest mean 896-token loss, i.e. retention of B2's original-context ability;
- `log.jsonl`.

Both best files use the Phase-1 `best.pt` format, so `phase3/backbones.py` loads them.

## Setup: GPUs and attached inputs

In [ ]:
import os, sys, glob, json, time, subprocess
SESSION_START = time.time()
WORK = "/kaggle/working"
os.chdir(WORK); sys.path.insert(0, WORK)
import torch
N_GPUS = torch.cuda.device_count()
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPUS)])
assert N_GPUS == 2, "use Accelerator GPU T4 x2 (the run must use the same GPU count in every session)"

def find(pattern):
    hits = sorted(glob.glob(os.path.join("/kaggle/input", "**", pattern), recursive=True))
    print(f"{pattern}: {hits if hits else 'NOT FOUND'}")
    return hits
NEW = {s: find(f"ctxdata/new_{s}.bin") for s in ("high_courts", "tribunals")}
UNIQ = {s: find(f"ctxdata/{s}_unique.bin") for s in ("supreme_court", "acts")}
HELD = {s: find(f"ctxdata/heldout_{s}.bin") for s in ("high_courts", "tribunals")}
MANIFEST = find("ctxdata/manifest_ctxext.json")
ORIG_TRAIN = [p for p in find("train.bin") if "/ctxdata/" not in p]
ORIG_VAL = [p for p in find("val.bin") if "/ctxdata/" not in p]
B2 = find("run_B2/best.pt")
PREVIOUS = find("ctxext/last.pt")
assert all(NEW.values()) and all(UNIQ.values()) and all(HELD.values()) and MANIFEST, "attach the capstone_b2_ctxdata output"
assert ORIG_TRAIN and ORIG_VAL and B2, "attach the Phase-1 data output (train.bin, val.bin) and the B2 output (run_B2/best.pt)"
man = json.load(open(MANIFEST[0]))
assert man["replay"].get("exact_match") is True, "the data audit's replay did not match the original train.bin: do not train on it"
print("data audit: replay verified | new training tokens available:", man["available_new_training_tokens"] / 1e9, "B")
print("resuming from a previous session" if PREVIOUS else "no previous ctxext/last.pt attached: session 1")

## Code files

In [ ]:
%%writefile arch3.py
# Architecture 2 option — rope.py + attention3.py + decoder3.py (your code: GQA + RoPE)
# ======== rope.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Rotary Positional Embeddings, implemented from scratch."""
import torch


def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """
    Returns (cos, sin) tensors of shape (max_seq_len, d_head) ready to be applied to
    query/key tensors of head dimension d_head. d_head must be even. This scaffold uses
    the split-half rotate_half convention: channel i is paired with channel i+d_head/2.
    """
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    # TODO: compute inverse frequencies: 1 / base^(2i/d_head) for i in [0, d_head/2)
    inverse = [1/(base**(2*i/d_head)) for i in range(0,d_head//2)]
    inverse = torch.tensor(inverse, dtype=torch.float32)  # FIX (Claude): torch.outer needs a tensor, not a list
    # TODO: outer product of positions [0..max_seq_len) with inverse frequencies -> (T, d_head/2)
    pos = torch.arange(max_seq_len, dtype=torch.float32)  # FIX (Claude): float, to match `inverse` in torch.outer
    outer = torch.outer(pos,inverse)
    # TODO: construct angles = torch.cat([freqs, freqs], dim=-1) so the shape
    #       becomes (T, d_head) and matches the split-half rotate_half convention.
    #       Do NOT use repeat_interleave here; it produces a different channel layout.
    #       Then return angles.cos(), angles.sin().
    angles = torch.cat([outer,outer],dim=-1)
    return angles.cos() , angles.sin()
    


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree
    rotation step used by the "rotate_half" formulation of RoPE."""
    x1,x2 = torch.chunk(x,2,dim=-1)
    x2=-x2
    x = torch.cat([x2,x1],dim=-1)
    return x



def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: (B, n_heads, T, d_head) -- a query or key tensor.
    cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T.
    Returns the rotated tensor, same shape as x.
    Formula: x_rotated = x * cos + rotate_half(x) * sin
    """
    # TODO: broadcast cos/sin to (1, 1, T, d_head) and apply the formula above
    cos = torch.unsqueeze(cos,dim=0)
    cos = torch.unsqueeze(cos,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    x_rotated = x*cos+rotate_half(x)*sin
    
    return x_rotated


# ======== attention3.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale
       
        attention_scores = attention_scores.masked_fill(
            ~self.causal_mask[:T,:T],float("-inf")
        )
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output


# ======== decoder3.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Decoder-only Transformer LM -- Block now passes use_rope through to attention, and
DecoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE  # unchanged from Question 1


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg=cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x  # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")

In [ ]:
%%writefile fastattn.py
# Plumbing: fused SDPA attention + KV cache + generation around your modules
"""Efficiency plumbing around the hand-written A3 models (written with Claude; declare it).

Nothing here changes an architecture: parameters, shapes and the maths of every model stay
exactly as defined in arch1/arch2/arch3. `patch_model` swaps each attention module's forward
for one that computes the *same* function with fused kernels, and adds an optional KV cache
for generation. `check_equivalence` in the notebook verifies outputs match the original code.

* Training: F.scaled_dot_product_attention(is_causal=True). On T4 (sm_75) PyTorch cannot use
  the FlashAttention-2 kernel (needs sm_80+); it dispatches to the memory-efficient fused
  kernel, which gives the same benefit: no (T x T) score matrix materialised in HBM.
* Generation: per-layer KV cache (keys/values for MHA/GQA, the compressed latent for MLA),
  prefill once, then one token per step.
"""
import sys
import types
from contextlib import contextmanager

import torch
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# attention forward (drop-in for all three hand-written attention classes)
# ---------------------------------------------------------------------------

def _sdpa(q, k, v, causal, dropout_p, scale):
    return F.scaled_dot_product_attention(q, k, v, is_causal=causal, dropout_p=dropout_p, scale=scale)


def _fast_attention_forward(self, x):
    B, T, C = x.shape
    cache = getattr(self, "_kv_cache", None)
    start = cache["pos"] if cache is not None else 0
    n_heads, d_head = self.n_heads, self.d_head
    dropout_p = self.attn_dropout.p if self.training else 0.0

    q = self.q_proj(x).view(B, T, n_heads, d_head).transpose(1, 2)

    if hasattr(self, "kv_down_proj"):                       # MLA: cache the shared latent
        latent = self.kv_down_proj(x)
        if cache is not None:
            cache["latent"][:, start:start + T] = latent
            latent = cache["latent"][:, :start + T]
        S = latent.shape[1]
        k = self.k_up_proj(latent).view(B, S, n_heads, d_head).transpose(1, 2)
        v = self.v_up_proj(latent).view(B, S, n_heads, d_head).transpose(1, 2)
    else:                                                   # MHA / GQA (+ optional RoPE)
        n_kv = getattr(self, "n_kv_heads", n_heads)
        k = self.k_proj(x).view(B, T, n_kv, d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, n_kv, d_head).transpose(1, 2)
        if getattr(self, "use_rope", False):
            cos = self.rope_cos[start:start + T]
            sin = self.rope_sin[start:start + T]
            q = self._apply_rope(q.float(), cos, sin).to(v.dtype)
            k = self._apply_rope(k.float(), cos, sin).to(v.dtype)
        if cache is not None:
            cache["k"][:, :, start:start + T] = k
            cache["v"][:, :, start:start + T] = v
            k = cache["k"][:, :, :start + T]
            v = cache["v"][:, :, :start + T]
        n_rep = n_heads // n_kv
        if n_rep > 1:
            k = k.repeat_interleave(n_rep, dim=1)
            v = v.repeat_interleave(n_rep, dim=1)

    # prefill (start == 0) is causal; a single decode step attends to the whole cache
    causal = T > 1
    if causal and start > 0:
        raise ValueError("multi-token steps are only supported from an empty cache")
    y = _sdpa(q, k, v, causal, dropout_p, self.attention_scale)
    y = y.transpose(1, 2).reshape(B, T, n_heads * d_head)
    return self.resid_dropout(self.out_proj(y))


def attention_modules(model):
    return [block.attn for block in model.blocks]


def patch_model(model):
    """Swap every attention forward for the fused/cached version (same parameters)."""
    for attn in attention_modules(model):
        attn._kv_cache = None
        if getattr(attn, "use_rope", False):
            # the user's own apply_rope, from the module that defines this attention class
            attn._apply_rope = sys.modules[type(attn).__module__].apply_rope
        attn.forward = types.MethodType(_fast_attention_forward, attn)
    return model


# ---------------------------------------------------------------------------
# KV cache
# ---------------------------------------------------------------------------

@contextmanager
def kv_cache(model, batch_size, max_len, dtype):
    """Allocate a per-layer cache. Inside the context call `step(model, ids)` repeatedly."""
    dev = next(model.parameters()).device
    caches = []
    for attn in attention_modules(model):
        c = {"pos": 0}
        if hasattr(attn, "kv_down_proj"):
            c["latent"] = torch.empty(batch_size, max_len, attn.kv_latent_dim, device=dev, dtype=dtype)
        else:
            n_kv = getattr(attn, "n_kv_heads", attn.n_heads)
            shape = (batch_size, n_kv, max_len, attn.d_head)
            c["k"] = torch.empty(shape, device=dev, dtype=dtype)
            c["v"] = torch.empty(shape, device=dev, dtype=dtype)
        attn._kv_cache = c
        caches.append(c)
    full_pe = getattr(model, "pos_emb", None)
    model._fast_state = {"caches": caches, "full_pe": full_pe}
    try:
        yield
    finally:
        for attn in attention_modules(model):
            attn._kv_cache = None
        if full_pe is not None:
            model.pos_emb = full_pe
        del model._fast_state


def step(model, ids):
    """Run `ids` (B, T) through the model at the current cache position; returns logits (B, T, V)."""
    state = model._fast_state
    pos = state["caches"][0]["pos"]
    if state["full_pe"] is not None:
        # DecoderLM.forward adds self.pos_emb[:T]; shift the table so rows pos..pos+T are used
        model.pos_emb = state["full_pe"][pos:]
    logits = model(ids)
    for c in state["caches"]:
        c["pos"] += ids.shape[1]
    return logits


# ---------------------------------------------------------------------------
# sampling + word-budgeted generation
# ---------------------------------------------------------------------------

def sample_next(logits, temperature, top_k, top_p, banned_ids, prev_ids=None, repetition_penalty=1.0):
    logits = torch.nan_to_num(logits.float(), nan=-1e4)   # a diverged checkpoint must not crash sampling
    if banned_ids is not None:
        logits[:, banned_ids] = -float("inf")
    if repetition_penalty != 1.0 and prev_ids is not None:
        seen = torch.gather(logits, 1, prev_ids)
        seen = torch.where(seen > 0, seen / repetition_penalty, seen * repetition_penalty)
        logits = logits.scatter(1, prev_ids, seen)
    if not temperature:
        return logits.argmax(-1)
    logits = logits / temperature
    if top_k:
        kth = torch.topk(logits, top_k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p and top_p < 1.0:
        sorted_logits, idx = torch.sort(logits, descending=True)
        probs = sorted_logits.softmax(-1)
        drop = probs.cumsum(-1) - probs > top_p          # keep the smallest set with mass >= top_p
        sorted_logits = sorted_logits.masked_fill(drop, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, idx, sorted_logits)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(1)


def ngram_ban(seq, n, vocab):
    """(B, V) bool: tokens that would complete an n-gram (in token ids) already present in `seq` (B, S).
    For each row, every earlier occurrence of the last n-1 tokens bans the token that followed it."""
    B, S = seq.shape
    if n < 2 or S < n:
        return torch.zeros(B, vocab, dtype=torch.bool, device=seq.device)
    grams = seq.unfold(1, n, 1)                                   # (B, S-n+1, n)
    hit = (grams[:, :, :-1] == seq[:, None, S - n + 1:]).all(-1)  # earlier (n-1)-gram == current suffix
    cnt = torch.zeros(B, vocab, device=seq.device).scatter_add_(1, grams[:, :, -1], hit.float())
    return cnt > 0


@torch.inference_mode()
def generate_words(model, tokenizer, prompts, *, max_seq_len, target_words, max_new_tokens,
                   batch_size, temperature, top_k, top_p, repetition_penalty, device, amp_dtype,
                   log_every=10, no_repeat_ngram=0, ngram_scope="output"):
    """no_repeat_ngram = n > 1 forbids repeating any n-gram of TOKEN ids (HF no_repeat_ngram_size), counted over
    the generated text only (ngram_scope="output") or over prompt + generated text ("all")."""
    assert ngram_scope in ("output", "all"), ngram_scope
    """Continue each prompt by `target_words` whitespace words. Returns list[str] in input order.

    Prompts are bucketed by token length so a batch needs no padding: every prompt in a batch
    keeps its last L tokens, L = shortest prompt in the batch (capped so prompt + new tokens fit
    max_seq_len). Words are counted from the tokenizer's end-of-word flag, so we stop as soon as
    every row has produced target_words complete words.
    """
    model.eval()
    eow = torch.tensor(tokenizer.token_eow, device=device, dtype=torch.bool)
    has_bytes = torch.tensor([len(b) > 0 for b in tokenizer.token_bytes], device=device, dtype=torch.bool)
    banned = torch.arange(tokenizer.n_special, device=device)     # never emit pad/unk/</s>/...
    encoded = [tokenizer.encode(p) for p in prompts]
    order = sorted(range(len(prompts)), key=lambda i: len(encoded[i]))
    max_prompt = max_seq_len - max_new_tokens
    assert max_prompt > 0, "max_new_tokens must be smaller than max_seq_len"
    results = [None] * len(prompts)

    for b, start in enumerate(range(0, len(order), batch_size)):
        idxs = order[start:start + batch_size]
        L = min(min(len(encoded[i]) for i in idxs), max_prompt)
        ids = torch.tensor([encoded[i][-L:] for i in idxs], device=device, dtype=torch.long)
        B = len(idxs)
        words = torch.zeros(B, dtype=torch.int32, device=device)
        in_word = torch.zeros(B, dtype=torch.bool, device=device)     # prompts end on a full word
        done = torch.zeros(B, dtype=torch.bool, device=device)
        out = []
        with torch.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"), \
                kv_cache(model, B, L + max_new_tokens, amp_dtype if device.type == "cuda" else torch.float32):
            logits = step(model, ids)[:, -1]
            history = ids
            for t in range(max_new_tokens):
                # a bare </w> right after a finished word never occurs in encoded text: forbid it
                logits = logits.float().clone()
                logits[~in_word, tokenizer.eow_id] = -float("inf")
                logits[:, len(tokenizer.token_bytes):] = -float("inf")   # ids the tokenizer can't decode
                if no_repeat_ngram and no_repeat_ngram > 1:
                    seq = history if ngram_scope == "all" else (torch.stack(out, 1) if out else ids[:, :0])
                    ban = ngram_ban(seq, no_repeat_ngram, logits.size(1))
                    ban &= ~(ban | torch.isinf(logits)).all(-1, keepdim=True)  # never leave a row with no option
                    logits = logits.masked_fill(ban, -float("inf"))
                nxt = sample_next(logits, temperature, top_k, top_p, banned,
                                  history if repetition_penalty != 1.0 else None, repetition_penalty)
                out.append(nxt)
                # a word ends on an end-of-word token that closes something non-empty
                open_ = in_word | has_bytes[nxt]
                words += (eow[nxt] & open_ & ~done).int()
                in_word = open_ & ~eow[nxt]
                done |= words >= target_words
                if bool(done.all()):
                    break
                if repetition_penalty != 1.0 or (no_repeat_ngram and ngram_scope == "all"):
                    history = torch.cat([history, nxt[:, None]], 1)
                logits = step(model, nxt[:, None])[:, -1]
        gen = torch.stack(out, 1).tolist()
        for row, i in enumerate(idxs):
            text = tokenizer.decode(gen[row])
            results[i] = " ".join(text.split()[:target_words])
        if log_every and (b + 1) % log_every == 0:
            print(f"  generated {min(start + batch_size, len(order))}/{len(order)}", flush=True)
    return results


# ---------------------------------------------------------------------------
# verification helpers (run in the notebook before training)
# ---------------------------------------------------------------------------

@torch.no_grad()
def check_equivalence(make_model, vocab_size, seq_len=48, rtol=1e-4):
    """Original forward vs patched forward vs KV-cached incremental decoding, fp32 on CPU.
    Returns max |difference| relative to the largest |logit| of the original model."""
    torch.manual_seed(0)
    ref = make_model().eval()
    fast = make_model().eval()
    fast.load_state_dict(ref.state_dict())
    patch_model(fast)
    idx = torch.randint(0, vocab_size, (2, seq_len))
    a = ref(idx)
    scale = a.abs().max().item()
    err_full = (a - fast(idx)).abs().max().item() / scale
    with kv_cache(fast, 2, seq_len, torch.float32):
        pre = step(fast, idx[:, :seq_len // 2])
        steps = [step(fast, idx[:, t:t + 1]) for t in range(seq_len // 2, seq_len)]
    err_cache = (a - torch.cat([pre] + steps, 1)).abs().max().item() / scale
    assert err_full < rtol and err_cache < rtol, (err_full, err_cache)
    return err_full, err_cache

In [ ]:
%%writefile nr_sampler.py
"""Deterministic no-repeat window sampler for train_ctxext.py (plumbing; declare it).

Windows
  A token file (uint16, documents separated by </s> exactly as the data build wrote them) of n tokens is cut into
  n_w = (n - 1) // L fixed windows; window w is tokens [w*L, w*L + L + 1): inputs w*L .. w*L+L-1, targets one to the
  right. Input positions of different windows never overlap, so every token position of a file is an input at most once
  and a target at most once. The </s> separators stay where the build put them; a window may span a document boundary
  (packed training, as in Phase 1). The last (n - 1) - n_w*L tokens of a file are never used.

Order (fixed by seed, file lengths, weights and budget; identical on every rank and in every session)
  * budget: n_windows = max_tokens // L (whole windows only; never more than max_tokens input tokens).
  * allocation: windows per file proportional to the weights (largest remainder); a file that has fewer windows than
    its share gives everything it has and the rest is re-allocated to the other files in proportion to their weights.
    If all files together have fewer windows than the budget: error (would need repeats).
  * per file: the first n_alloc entries of a seeded permutation of its window ids (each id at most once).
  * source interleaving: a seeded permutation of the multiset of source labels, so the mixture is uniform over the run.
  The global order is the array pair (src[k], wid[k]), k = 0 .. n_windows-1. Step s consumes positions
  [s*S, min((s+1)*S, n_windows)), S = windows per optimiser step; rank r takes the r-th block of per_rank positions.
"""
import hashlib

import numpy as np


def allocate(avail, weights, n_total):
    """windows per file: proportional to weights, capped by availability, remainder redistributed."""
    avail = np.asarray(avail, np.int64)
    w = np.asarray(weights, float)
    assert (w > 0).all() and len(w) == len(avail)
    if avail.sum() < n_total:
        raise ValueError(f"budget needs {n_total} windows but the files have only {int(avail.sum())}: lower max_tokens "
                         "or add data (the no-repeat sampler never repeats a window)")
    alloc = np.zeros(len(avail), np.int64)
    open_ = np.ones(len(avail), bool)
    left = n_total
    while left > 0:
        ww = np.where(open_, w, 0.0)
        share = ww / ww.sum() * left
        base = np.floor(share).astype(np.int64)
        rem = left - base.sum()
        order = np.argsort(-(share - base), kind="stable")    # largest remainder, ties by file order
        for i in order[:rem]:
            base[i] += 1
        room = avail - alloc
        take = np.minimum(base, room)
        alloc += take
        left -= int(take.sum())
        open_ &= (avail - alloc) > 0
    assert alloc.sum() == n_total and (alloc <= avail).all()
    return alloc


def build_order(file_lengths, weights, max_tokens, L, seed):
    n_avail = np.array([(n - 1) // L for n in file_lengths], np.int64)
    n_total = int(max_tokens // L)
    alloc = allocate(n_avail, weights, n_total)
    wid = np.empty(n_total, np.int64)
    src = np.repeat(np.arange(len(alloc), dtype=np.int8), alloc)
    src = src[np.random.default_rng([seed, 1_000_003]).permutation(n_total)]
    for f, a in enumerate(alloc):
        perm = np.random.default_rng([seed, 7_919, f]).permutation(n_avail[f])[:a]
        wid[src == f] = perm
    order = {"src": src, "wid": wid, "n_avail": n_avail, "alloc": alloc, "L": int(L), "seed": int(seed),
             "max_tokens": int(max_tokens), "file_lengths": np.asarray(file_lengths, np.int64)}
    check_order(order)
    return order


def check_order(order):
    """assertions: every window id valid and used at most once per file; counts match the allocation."""
    src, wid = order["src"], order["wid"]
    assert len(src) == len(wid) == int(order["alloc"].sum())
    for f, a in enumerate(order["alloc"]):
        ids = wid[src == f]
        assert len(ids) == a, (f, len(ids), a)
        assert len(ids) == 0 or (ids.min() >= 0 and ids.max() < order["n_avail"][f]), f"window id out of range in file {f}"
        assert len(np.unique(ids)) == len(ids), f"repeated window id in file {f}"


def order_digest(order):
    h = hashlib.sha256()
    for k in ("src", "wid", "alloc", "file_lengths"):
        h.update(np.ascontiguousarray(order[k]).tobytes())
    h.update(repr((order["L"], order["seed"], order["max_tokens"])).encode())
    return h.hexdigest()


def step_positions(step, S, n_total, world, rank):
    """global order positions this rank trains on at this step (may be empty in a final partial step)."""
    per_rank = S // world
    lo = step * S + rank * per_rank
    return np.arange(max(lo, 0), max(min(lo + per_rank, n_total, (step + 1) * S), lo), dtype=np.int64)


def per_source_counts(order, upto):
    """windows consumed per file among order positions [0, upto)."""
    return np.bincount(order["src"][:upto], minlength=len(order["alloc"])).astype(np.int64)

In [ ]:
%%writefile train_ctxext.py
#!/usr/bin/env python3
"""Continued pretraining of the Phase-1 B2 model with an extended context (plumbing; declare it).

  torchrun --nproc_per_node=2 train_ctxext.py config.json        (Kaggle 2xT4)
  python train_ctxext.py config.json                              (1 GPU / CPU smoke test)
  python train_ctxext.py config.json --probe 30                   (30 optimiser steps: speed, memory, ETA, then exit)

What it does
  * builds B2's architecture (your arch3.py) with model.max_seq_len = cfg["max_seq_len"] (e.g. 1024 instead of 896) and
    loads B2's weights. The RoPE tables and causal mask are non-persistent buffers, so the saved weights do not depend on
    the sequence length; they are rebuilt here for the new length using cfg["rope_extension"] (your choice):
      "extrapolate": positions 0..L-1 as trained (positions >= 896 were never seen in pretraining)
      "interpolate": positions scaled by old/new length, so 0..L-1 map into the trained range 0..895 (position interpolation)
      "ntk":         RoPE base enlarged by (new/old)^(d_head/(d_head-2)) ("NTK-aware" scaling)
  * trains on fixed, non-overlapping windows of length L+1 from one or more uint16 token files, each window used at most
    once, in an order fixed by the seed (nr_sampler.py: allocation, permutation, per-step / per-rank positions). The order
    and the cursor are stored in last.pt, so a resumed run neither repeats nor skips windows.
  * learning-rate schedule driven by TOKENS (not wall-clock), so it is identical however the run is split into sessions:
    linear warm-up over warmup_steps, then cosine (or linear) decay to min_lr at max_tokens.
  * full checkpoint last.pt (model + optimiser + scaler + step + tokens + RNG) every ckpt_every_min minutes, at the
    session limit (session_hours) and on SIGTERM; best.pt (weights only, same format as Phase-1 best.pt, so the Phase 2/3
    pipelines can load it) at every evaluation that improves the validation loss at the new length.
  * resume: the newest last.pt (most tokens) found in out_dir or under any of cfg["resume_search"] globs
    (on Kaggle: the previous session's output attached as an input, from any account).
  * evaluation on every set in cfg["eval_sets"] ({name: uint16 .bin}): loss over windows of the new length L, split into
    positions < old length and >= old length, and loss over windows of the old length (896). Two selections are kept:
      best_newctx.pt  lowest mean (over eval sets) loss at the new length
      best_oldctx.pt  lowest mean loss at the old length (retention of B2's original-context ability)
    both in the Phase-1 best.pt format (loadable by phase3/backbones.py). The step-0 evaluation is B2 itself (baseline).
  * training mixture: cfg["train_files"] = [{"path": ..., "weight": w | None}]; the max_tokens // L windows of the run are
    split over the files in proportion to w (None for all = proportional to the windows each file has), capped by what a
    file has. The token budget is never exceeded: whole windows only, and a final partial step if the count needs one.
  * session guard: world size, batch_tokens, tokens/step, max_tokens, max_seq_len and rope method are stored in last.pt;
    a resume with any of them different stops with an error (that would silently change the token schedule).
  * RNG: torch CPU and per-rank CUDA generator states (dropout) are saved and restored.
"""
import argparse
import glob
import importlib
import json
import math
import os
import signal
import sys
import time
from datetime import timedelta

import numpy as np
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parallel import DistributedDataParallel as DDP

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import fastattn  # noqa: E402
import nr_sampler  # noqa: E402

ap = argparse.ArgumentParser()
ap.add_argument("config")
ap.add_argument("--probe", type=int, default=0, help="run N optimiser steps, report speed/memory/ETA, exit without saving")
args = ap.parse_args()
cfg = json.load(open(args.config))
REQUIRED = {"": ["init_from", "train_files", "eval_sets", "out_dir", "max_seq_len", "rope_extension"],
            "train": ["max_tokens", "batch_tokens", "lr", "min_lr", "warmup_steps", "weight_decay", "betas", "grad_clip",
                      "schedule", "seed", "eval_every_min", "ckpt_every_min", "session_hours", "eval_windows"]}
missing = [f"{s + '.' if s else ''}{k}" for s, ks in REQUIRED.items() for k in ks if (cfg.get(s, {}) if s else cfg).get(k) is None]
assert not missing, "fill these config values first (your choices): " + ", ".join(missing)
assert cfg["rope_extension"] in ("extrapolate", "interpolate", "ntk")
TR, OUT = cfg["train"], cfg["out_dir"]
os.makedirs(OUT, exist_ok=True)

# ---------------- distributed / device
WORLD, RANK, LOCAL = int(os.environ.get("WORLD_SIZE", 1)), int(os.environ.get("RANK", 0)), int(os.environ.get("LOCAL_RANK", 0))
DIST, CUDA, MASTER = WORLD > 1, torch.cuda.is_available(), int(os.environ.get("RANK", 0)) == 0
if CUDA:
    torch.cuda.set_device(LOCAL)
device = torch.device(f"cuda:{LOCAL}" if CUDA else "cpu")
if DIST:
    dist.init_process_group("nccl" if CUDA else "gloo", timeout=timedelta(minutes=15))
# bf16 only with native bf16 tensor cores (compute capability >= 8.0). is_bf16_supported() is also True on a T4, where
# bf16 is emulated and several times slower; the T4 uses fp16 + GradScaler, like the Phase-1 B / B2 trainer.
AMP = (torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16) if CUDA else torch.float32
USE_SCALER = CUDA and AMP == torch.float16
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
SESSION_T0 = time.time()


def log(msg, **kv):
    if not MASTER:
        return
    rec = {"t": round(time.time(), 1), "msg": msg, **kv}
    print(json.dumps(rec), flush=True)
    with open(os.path.join(OUT, "log.jsonl"), "a") as f:
        f.write(json.dumps(rec) + "\n")


# ---------------- model: B2 architecture at the new length + B2 weights
src = torch.load(cfg["init_from"], map_location="cpu", weights_only=False)
scfg = src["config"]
OLD_LEN = scfg["model"]["max_seq_len"]
L = cfg["max_seq_len"]
mdict = dict(scfg["model"], max_seq_len=L)
if cfg.get("dropout") is not None:
    mdict["dropout"] = cfg["dropout"]
arch = importlib.import_module(scfg["module"])
torch.manual_seed(TR["seed"])
raw = arch.DecoderLM(arch.LMConfig(**mdict))


def clean(sd):
    return {k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in sd.items()}


def set_rope(model):
    """rebuild every attention module's RoPE tables for length L with the chosen extension method."""
    for blk in model.blocks:
        a = blk.attn
        if not getattr(a, "use_rope", False):
            continue
        d = a.d_head
        base, scale = 10000.0, 1.0
        if cfg["rope_extension"] == "interpolate":
            scale = OLD_LEN / L
        elif cfg["rope_extension"] == "ntk":
            base = 10000.0 * (L / OLD_LEN) ** (d / (d - 2))
        inv = 1.0 / (base ** (torch.arange(0, d // 2, dtype=torch.float32) * 2 / d))
        pos = torch.arange(L, dtype=torch.float32) * scale
        ang = torch.outer(pos, inv)
        ang = torch.cat([ang, ang], -1)                          # split-half convention, as in your rope.py
        a.rope_cos = ang.cos().to(a.rope_cos.device)
        a.rope_sin = ang.sin().to(a.rope_sin.device)


missing_keys, unexpected = raw.load_state_dict(clean(src["model"]), strict=False)
assert not unexpected and not [k for k in missing_keys if "rope" not in k and "mask" not in k], (missing_keys, unexpected)
set_rope(raw)
n_params = sum(p.numel() for p in raw.parameters())
fastattn.patch_model(raw)
raw.to(device)
del src

# ---------------- data
trains = [np.memmap(f["path"], dtype=np.uint16, mode="r") for f in cfg["train_files"]]
flens = [len(m) for m in trains]
assert all(n > L + 1 for n in flens), "a train file is shorter than one window"
wts = [f.get("weight") for f in cfg["train_files"]]
assert all(w is None for w in wts) or all(w is not None and w > 0 for w in wts), "give a weight for every train file, or none"
weights = [(n - 1) // L for n in flens] if wts[0] is None else wts
ORDER = nr_sampler.build_order(flens, weights, TR["max_tokens"], L, TR["seed"])   # identical on every rank / session
N_WIN = len(ORDER["src"])
DIGEST = nr_sampler.order_digest(ORDER)
evals = {k: np.memmap(v, dtype=np.uint16, mode="r") for k, v in cfg["eval_sets"].items()}
assert all(len(m) > L + 1 for m in evals.values()), "an eval set is shorter than one window"
V = mdict["vocab_size"]
assert max(int(m[:2_000_000].max()) for m in trains) < V, "token id >= vocab_size: data / tokenizer mismatch"


def windows(pairs):
    """(file, window id) pairs -> input / target tensors; window w = tokens [w*L, w*L + L + 1)."""
    rows = [trains[f][w * L:w * L + L + 1] for f, w in pairs]
    assert all(len(r) == L + 1 for r in rows)
    t = torch.from_numpy(np.stack(rows).astype(np.int64))
    return t[:, :-1].to(device, non_blocking=True), t[:, 1:].to(device, non_blocking=True)


def dummy_batch(n):
    """first n windows of file 0: memory search, and the zero-weight filler of an empty micro-batch in a final partial step."""
    return windows([(0, i % ORDER["n_avail"][0]) for i in range(n)])


def loss_of(model, x, y, reduce=True):
    with torch.autocast("cuda", dtype=AMP, enabled=CUDA):
        logits = model(x)
    l = F.cross_entropy(logits.float().view(-1, V), y.reshape(-1), reduction="mean" if reduce else "none")
    return l


# ---------------- batch geometry: global batch_tokens split over GPUs, largest micro-batch that fits
seqs = max(WORLD, TR["batch_tokens"] // L)
seqs -= seqs % WORLD
per_rank = seqs // WORLD
mb = TR.get("micro_batch")
if mb is None:
    mb = 1
    for cand in sorted([d for d in range(1, per_rank + 1) if per_rank % d == 0], reverse=True):
        if not CUDA:
            mb = 1; break
        try:
            torch.cuda.empty_cache()
            x, y = dummy_batch(cand)
            loss_of(raw, x, y).backward(); raw.zero_grad(set_to_none=True)
            if torch.cuda.max_memory_allocated() < 0.80 * torch.cuda.get_device_properties(device).total_memory:
                mb = cand; break
        except torch.cuda.OutOfMemoryError:
            raw.zero_grad(set_to_none=True)
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    if DIST:
        t = torch.tensor([mb], device=device); dist.all_reduce(t, op=dist.ReduceOp.MIN); mb = int(t.item())
assert per_rank % mb == 0
ACC = per_rank // mb
TOK_PER_STEP = seqs * L                                       # a full step; the final step may be partial
TOTAL_STEPS = math.ceil(N_WIN / seqs)
assert N_WIN * L <= TR["max_tokens"]
ddp = DDP(raw, device_ids=[LOCAL] if CUDA else None, broadcast_buffers=False, gradient_as_bucket_view=True) if DIST else None
base = ddp if DIST else raw
model = base
COMPILED = False
if TR.get("compile", False) and CUDA:                        # as in the Phase-1 refine trainer; eager fallback
    t_c = time.time()
    try:
        model = torch.compile(base)
        x, y = dummy_batch(mb)
        loss_of(model, x, y).backward(); raw.zero_grad(set_to_none=True)
        COMPILED = True
        log("torch.compile ok", seconds=round(time.time() - t_c, 1))
    except Exception as e:
        model = base; raw.zero_grad(set_to_none=True)
        log("torch.compile failed, using eager", error=repr(e)[:300])

decay = [p for p in raw.parameters() if p.dim() >= 2]
no_decay = [p for p in raw.parameters() if p.dim() < 2]
groups = [{"params": decay, "weight_decay": TR["weight_decay"]}, {"params": no_decay, "weight_decay": 0.0}]
try:
    opt = torch.optim.AdamW(groups, lr=TR["lr"], betas=tuple(TR["betas"]), fused=CUDA)
except (RuntimeError, TypeError):
    opt = torch.optim.AdamW(groups, lr=TR["lr"], betas=tuple(TR["betas"]))
scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)

# ---------------- resume
step, tokens, history = 0, 0, []
best = {"newctx": float("inf"), "oldctx": float("inf")}
GUARD = {"world_size": WORLD, "batch_tokens": TR["batch_tokens"], "tokens_per_step": TOK_PER_STEP, "max_tokens": TR["max_tokens"],
         "max_seq_len": L, "rope_extension": cfg["rope_extension"], "warmup_steps": TR["warmup_steps"], "lr": TR["lr"],
         "min_lr": TR["min_lr"], "schedule": TR["schedule"], "sampler": "no_repeat", "sampler_order_sha256": DIGEST}
cands = [os.path.join(OUT, "last.pt")] + [p for g in cfg.get("resume_search", []) for p in glob.glob(g, recursive=True)]
found = []
for p in cands:
    if os.path.exists(p):
        try:
            meta = torch.load(p, map_location="cpu", weights_only=False)
            if meta.get("kind") == "ctxext_last" and meta["max_seq_len"] == L:
                found.append((meta["tokens"], p, meta))
        except Exception as e:
            log("skipping unreadable checkpoint", path=p, error=repr(e)[:200])
if found:
    tokens, path, ck = max(found, key=lambda x: x[0])
    diff = {k: (ck.get("guard", {}).get(k), v) for k, v in GUARD.items() if ck.get("guard", {}).get(k) != v}
    assert not diff, f"resume would change the run's schedule/geometry (checkpoint value, now): {diff}"
    raw.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
    step, best, history = ck["step"], ck["best"], ck.get("history", [])
    sp = ck["sampler"]
    assert sp["seed"] == TR["seed"] and np.array_equal(sp["src"], ORDER["src"]) and np.array_equal(sp["wid"], ORDER["wid"]), \
        "saved window order differs from the rebuilt one (data files, weights, seed or budget changed)"
    assert sp["cursor"] == min(step * seqs, N_WIN) and tokens == sp["cursor"] * L, (sp["cursor"], step, tokens)
    cpu_states = ck["rng"]["cpu"]
    torch.set_rng_state(cpu_states[RANK] if isinstance(cpu_states, list) else cpu_states)
    if CUDA and ck["rng"].get("cuda") and RANK < len(ck["rng"]["cuda"]):
        torch.cuda.set_rng_state(ck["rng"]["cuda"][RANK])
    log("resumed", path=path, step=step, tokens_B=round(tokens / 1e9, 4), windows_consumed=sp["cursor"],
        windows_remaining=N_WIN - sp["cursor"])
    del ck
log("setup", params_M=round(n_params / 1e6, 2), old_len=OLD_LEN, new_len=L, rope_extension=cfg["rope_extension"], gpus=WORLD,
    amp=str(AMP).split(".")[-1], compiled=COMPILED, seqs_per_step=seqs, micro_batch=mb, grad_accum=ACC, tokens_per_step=TOK_PER_STEP,
    total_steps=TOTAL_STEPS, start_step=step, target_tokens_B=round(TR["max_tokens"] / 1e9, 3))
log("sampler (no repeat)", files=[os.path.basename(f["path"]) for f in cfg["train_files"]], weights=list(map(float, weights)),
    windows_available=ORDER["n_avail"].tolist(), windows_allocated=ORDER["alloc"].tolist(),
    share_allocated=[round(a / N_WIN, 6) for a in ORDER["alloc"].tolist()], tokens_allocated=(ORDER["alloc"] * L).tolist(),
    budget_tokens=TR["max_tokens"], run_tokens=N_WIN * L, budget_tokens_unused=TR["max_tokens"] - N_WIN * L,
    final_step_windows=N_WIN - (TOTAL_STEPS - 1) * seqs, repeated_window_ids=0, order_sha256=DIGEST)


def data_progress(cursor):
    c = nr_sampler.per_source_counts(ORDER, cursor)
    return {"windows_consumed": int(cursor), "windows_remaining": int(N_WIN - cursor), "tokens_per_source": (c * L).tolist()}


def lr_at(s):
    if s < TR["warmup_steps"]:
        return TR["lr"] * (s + 1) / TR["warmup_steps"]
    p = min(1.0, (s - TR["warmup_steps"]) / max(1, TOTAL_STEPS - TR["warmup_steps"]))
    f = 0.5 * (1 + math.cos(math.pi * p)) if TR["schedule"] == "cosine" else 1 - p
    return TR["min_lr"] + (TR["lr"] - TR["min_lr"]) * f


@torch.no_grad()
def evaluate():
    """per eval set: new-length loss (all / positions < old / >= old) and old-length loss; plus their means over sets."""
    raw.eval()
    out = {}
    for name, val in evals.items():
        rng = np.random.default_rng(12345)                     # the same windows at every evaluation
        n = TR["eval_windows"]
        starts = rng.integers(0, len(val) - L - 1, n)
        tot_new = tot_lo = tot_hi = tot_old = 0.0
        for i in range(0, n, max(1, mb)):
            rows = np.stack([val[s:s + L + 1] for s in starts[i:i + mb]]).astype(np.int64)
            x, y = torch.from_numpy(rows[:, :-1]).to(device), torch.from_numpy(rows[:, 1:]).to(device)
            ln = loss_of(raw, x, y, reduce=False).view(x.shape[0], L)
            tot_new += ln.mean(1).sum().item(); tot_lo += ln[:, :OLD_LEN].mean(1).sum().item()
            tot_hi += ln[:, OLD_LEN:].mean(1).sum().item() if L > OLD_LEN else 0.0
            lo = loss_of(raw, x[:, :OLD_LEN], y[:, :OLD_LEN], reduce=False).view(x.shape[0], OLD_LEN)
            tot_old += lo.mean(1).sum().item()
        out[name] = {"new_len": tot_new / n, "pos_lt_old": tot_lo / n, "pos_ge_old": tot_hi / n if L > OLD_LEN else None, "old_len": tot_old / n}
    raw.train()
    out["mean"] = {k: float(np.mean([v[k] for v in out.values()])) for k in ("new_len", "old_len", "pos_lt_old")}
    if L > OLD_LEN:
        out["mean"]["pos_ge_old"] = float(np.mean([v["pos_ge_old"] for k, v in out.items() if k != "mean"]))
    return out


RNG_CUDA, RNG_CPU = None, None


def gather_rng():
    """every rank's CPU and CUDA generator states, collected on rank 0 (dropout masks differ per rank)."""
    global RNG_CUDA, RNG_CPU
    mine = (torch.get_rng_state(), torch.cuda.get_rng_state() if CUDA else None)
    if DIST:
        allst = [None] * WORLD
        dist.all_gather_object(allst, mine)
    else:
        allst = [mine]
    RNG_CPU = [a[0] for a in allst]
    RNG_CUDA = [a[1] for a in allst] if CUDA else None


def save_last():
    gather_rng()                                               # collective: every rank calls save_last
    if not MASTER:
        return
    st = {"kind": "ctxext_last", "model": raw.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(), "step": step,
          "tokens": tokens, "best": best, "history": history, "max_seq_len": L, "rope_extension": cfg["rope_extension"], "guard": GUARD,
          "rng": {"cpu": RNG_CPU, "cuda": RNG_CUDA},
          "sampler": {"kind": "no_repeat", "seed": TR["seed"], "cursor": min(step * seqs, N_WIN), "order_sha256": DIGEST,
                      "src": ORDER["src"], "wid": ORDER["wid"], "alloc": ORDER["alloc"], "n_avail": ORDER["n_avail"],
                      "file_lengths": ORDER["file_lengths"], "files": [f["path"] for f in cfg["train_files"]]}, "config": {"module": scfg["module"], "model": mdict, "tag": cfg.get("tag", "B2_ctxext")}}
    torch.save(st, os.path.join(OUT, "last.pt.tmp")); os.replace(os.path.join(OUT, "last.pt.tmp"), os.path.join(OUT, "last.pt"))


def save_best(kind, ev):
    """Phase-1 best.pt format (model + config with module/model) -> loadable by phase3/backbones.py."""
    if not MASTER:
        return
    name = f"best_{kind}.pt"
    st = {"model": raw.state_dict(), "config": {"module": scfg["module"], "model": mdict, "tag": cfg.get("tag", "B2_ctxext") + "_" + kind,
          "rope_extension": cfg["rope_extension"], "init_from": cfg["init_from"]}, "step": step, "tokens": tokens,
          "best_val": ev["mean"]["new_len" if kind == "newctx" else "old_len"], "eval": ev}
    torch.save(st, os.path.join(OUT, name + ".tmp")); os.replace(os.path.join(OUT, name + ".tmp"), os.path.join(OUT, name))


STOP = {"flag": False}
signal.signal(signal.SIGTERM, lambda *a: STOP.update(flag=True))

if step == 0 and not args.probe:
    ev = evaluate(); history.append({"step": 0, "tokens": 0, "eval": ev}); log("eval (B2 weights, before training = baseline)", eval=ev)
t_eval = t_ckpt = time.time()
# test-only: record every consumed window (step, order position, file, window id) per rank
REC = open(os.path.join(OUT, f"windows_rank{RANK}.txt"), "a") if cfg.get("debug_record_windows") and not args.probe else None
t_run, tok_run = time.time(), 0
end_step = step + args.probe if args.probe else TOTAL_STEPS
raw.train()
while step < end_step:
    lr = lr_at(step)
    for g in opt.param_groups:
        g["lr"] = lr
    tot = 0.0
    pos = nr_sampler.step_positions(step, seqs, N_WIN, WORLD, RANK)
    n_step = min(seqs, N_WIN - step * seqs)                   # windows in this step over all ranks (< seqs only at the end)
    assert n_step > 0
    for m in range(ACC):
        chunk = pos[m * mb:(m + 1) * mb]
        if len(chunk):
            x, y = windows(zip(ORDER["src"][chunk].tolist(), ORDER["wid"][chunk].tolist()))
            scale = len(chunk) * WORLD / n_step                # gradient = mean over every window of the step
        else:
            x, y = dummy_batch(1); scale = 0.0                 # keeps the DDP collectives aligned; contributes nothing
        if REC is not None and len(chunk):
            REC.write("".join(f"{step} {k} {ORDER['src'][k]} {ORDER['wid'][k]}\n" for k in chunk.tolist()))
        ctx = ddp.no_sync() if (DIST and m < ACC - 1) else torch.enable_grad()
        with ctx:
            l = loss_of(model, x, y) * scale
            scaler.scale(l).backward()
        tot += l.item()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(raw.parameters(), TR["grad_clip"])
    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
    step += 1; tokens += n_step * L; tok_run += n_step * L
    assert tokens <= TR["max_tokens"] and tokens == min(step * seqs, N_WIN) * L
    if not math.isfinite(tot):
        log("non-finite loss; stopping", step=step); break
    now = time.time()
    if MASTER and (step % 20 == 0 or args.probe):
        rate = tok_run / (now - t_run)
        log("train", step=step, tokens_B=round(tokens / 1e9, 4), loss=round(tot, 4), lr=lr, tok_s=round(rate),
            eta_h=round((N_WIN * L - tokens) / rate / 3600, 2), **data_progress(min(step * seqs, N_WIN)))
    if args.probe:
        continue
    # one synchronised decision per step (rank 0's clock, broadcast), so every rank evaluates / saves / stops together
    flags = torch.tensor([int((now - SESSION_T0) / 3600 > TR["session_hours"] or STOP["flag"]),
                          int((now - t_eval) / 60 > TR["eval_every_min"]), int((now - t_ckpt) / 60 > TR["ckpt_every_min"])],
                         device=device)
    if DIST:
        dist.broadcast(flags, src=0)
    session_over, eval_due, ckpt_due = (bool(x) for x in flags.tolist())
    if eval_due or step >= end_step or session_over:
        ev = evaluate(); history.append({"step": step, "tokens": tokens, "eval": ev}); t_eval = time.time()
        saved = []
        for kind, key in (("newctx", "new_len"), ("oldctx", "old_len")):
            if ev["mean"][key] < best[kind]:
                best[kind] = ev["mean"][key]; save_best(kind, ev); saved.append(kind)
        log("eval", step=step, tokens_B=round(tokens / 1e9, 4), saved=saved, eval=ev)
    if ckpt_due or step >= end_step or session_over:
        if REC is not None:
            REC.flush()
        save_last(); t_ckpt = time.time(); log("checkpoint", step=step, tokens_B=round(tokens / 1e9, 4), **data_progress(min(step * seqs, N_WIN)))
    if session_over:
        log("session limit reached: stopping (resume in the next session)", step=step, tokens_B=round(tokens / 1e9, 4)); break

if args.probe and MASTER:
    rate = tok_run / (time.time() - t_run)
    peak = torch.cuda.max_memory_allocated() / 1e9 if CUDA else None
    log("probe", tok_s=round(rate), peak_mem_GB_per_gpu=round(peak, 2) if peak else None,
        hours_for_target=round(TR["max_tokens"] / rate / 3600, 1),
        sessions_needed=round(N_WIN * L / rate / 3600 / TR["session_hours"], 1))
if not args.probe and step >= TOTAL_STEPS:
    log("DONE: all windows consumed", step=step, tokens=tokens, budget_tokens=TR["max_tokens"], best=best,
        **data_progress(min(step * seqs, N_WIN)))
if REC is not None:
    REC.close()
if DIST:
    dist.destroy_process_group()

## CONFIG (your choices)

In [ ]:
# =====================================================================================
# YOUR CHOICES (course rule 9) - values from your execution plan (2026-10-16). Only the mixture is still open.
# =====================================================================================
TRAIN_MIXTURE = {      # your choice (2026-10-17): new HC + Tribunal data only; Phase-1 SC / Acts / train.bin not repeated
    "new_high_courts": 0.538,
    "new_tribunals": 0.462,
    "original_train": 0.0,
    "sc_unique": 0.0,
    "acts_unique": 0.0,
}
# files: new_high_courts / new_tribunals (genuinely new, deduplicated); sc_unique / acts_unique (Phase-1 SC / Acts docs once);
#        original_train (the Phase-1 train.bin as built). Weight 0 = file not used.
assert TRAIN_MIXTURE is not None, "choose the training mixture (see the data manifest for available tokens per file)"
FILES = {"new_high_courts": NEW["high_courts"][0], "new_tribunals": NEW["tribunals"][0], "sc_unique": UNIQ["supreme_court"][0],
         "acts_unique": UNIQ["acts"][0], "original_train": ORIG_TRAIN[0]}
CFG = {
    "tag": "B2_ctxext1024",
    "init_from": B2[0],
    "train_files": [{"path": FILES[k], "weight": w} for k, w in TRAIN_MIXTURE.items() if w != 0],
    "eval_sets": {"sc_val": ORIG_VAL[0], "hc_heldout": HELD["high_courts"][0], "trib_heldout": HELD["tribunals"][0]},
    "out_dir": os.path.join(WORK, "ctxext"),
    "resume_search": ["/kaggle/input/**/ctxext/last.pt"],
    "max_seq_len": 1024,
    "rope_extension": "extrapolate",
    "train": {
        "max_tokens": 1_650_000_000,     # total additional tokens (all sessions)
        "batch_tokens": 8192,            # global tokens per optimiser step (8 x 1024: 4 sequences per T4)
        "micro_batch": None,             # largest that fits (searched)
        "lr": 5e-5, "min_lr": 5e-6, "warmup_steps": 1000, "schedule": "cosine",
        "weight_decay": 0.1, "betas": [0.9, 0.95], "grad_clip": 1.0,
        "seed": 1234,
        # session plumbing (not hyperparameters)
        "compile": True,                 # torch.compile, as in the Phase-1 refine trainer (eager fallback)
        "session_hours": 11.0, "eval_every_min": 60, "ckpt_every_min": 30, "eval_windows": 200,
    },
}
json.dump(CFG, open("ctxext_config.json", "w"), indent=2)
# no-repeat sampler: max_tokens // 1024 fixed windows, each used once (allocation per file is printed by the trainer at setup)
n_win = CFG["train"]["max_tokens"] // CFG["max_seq_len"]
steps = -(-n_win // (CFG["train"]["batch_tokens"] // CFG["max_seq_len"]))
print(f"1024 tokens, extrapolate | {n_win:,} windows = {n_win * CFG['max_seq_len']:,} tokens "
      f"({CFG['train']['max_tokens'] - n_win * CFG['max_seq_len']} of the budget unused) = {steps:,} steps")

## Optional probe (speed / memory / ETA)

In [ ]:
# Probe (session 1): 30 optimiser steps on both GPUs in memory (nothing is saved) -> tokens/s, peak memory per GPU,
# hours and 11-h sessions needed for max_tokens. Then the step-0 evaluation is NOT run; it runs at the start of training.
RUN_PROBE = True
if RUN_PROBE:
    subprocess.run(["torchrun", "--nproc_per_node=2", "train_ctxext.py", "ctxext_config.json", "--probe", "30"], check=True)

## Train (resumable)

In [ ]:
# Full training: only after the review. Resumes automatically from the newest attached ctxext/last.pt.
RUN_TRAINING = False
if RUN_TRAINING:
    p = subprocess.run(["torchrun", "--nproc_per_node=2", "train_ctxext.py", "ctxext_config.json"])
    print("exit code", p.returncode, "| session hours used", round((time.time() - SESSION_START) / 3600, 2))
else:
    print("RUN_TRAINING = False (probe-only session)")

## Summary

In [ ]:
# progress (all sessions): per eval set, loss at 1024 (positions <896 / >=896) and at 896 (retention); step 0 = B2 itself
import pandas as pd
logf = os.path.join(CFG["out_dir"], "log.jsonl")
if os.path.exists(logf):
    recs = [json.loads(l) for l in open(logf)]
    rows = []
    for r in recs:
        if r["msg"].startswith("eval"):
            for name, v in r["eval"].items():
                rows.append({"step": r.get("step", 0), "set": name, **v})
    if rows:
        display(pd.DataFrame(rows).pivot_table(index="step", columns="set", values=["new_len", "old_len", "pos_ge_old"]))
    print(sorted(os.listdir(CFG["out_dir"])))